In [1]:
import torchaudio

In [2]:
root_path = "/kaggle/input/datasets/garvs777/libri3mix"

In [3]:
dataset_train360 = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="train-360"
)
dataset_train100 = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="train-100"
)
dataset_dev = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="dev"
)
dataset_test = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="test"
)

In [4]:
import os, json
import soundfile as sf

def build_json(wav_dir, out_path):
    entries = []
    for fname in sorted(os.listdir(wav_dir)):
        if not fname.endswith(".wav"):
            continue
        fpath = os.path.join(wav_dir, fname)
        info = sf.info(fpath)
        entries.append([fpath, info.frames])
    with open(out_path, "w") as f:
        json.dump(entries, f, indent=2)
    print(".json file formed.")

In [ ]:
os.makedirs("/kaggle/working/json_dir/dev", exist_ok=True)
os.makedirs("/kaggle/working/json_dir/test", exist_ok=True)
os.makedirs("/kaggle/working/json_dir/train-100", exist_ok=True)

In [ ]:
base = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/dev"
out_path = "/kaggle/working/json_dir/dev"

# comment out if building for the first time

build_json(f"{base}/mix_clean", f"{out_path}/mix_clean.json")
build_json(f"{base}/s1", f"{out_path}/s1.json")
build_json(f"{base}/s2", f"{out_path}/s2.json")
build_json(f"{base}/s3", f"{out_path}/s3.json")

In [ ]:
base = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-100"
out_path = "/kaggle/working/json_dir/train-100"

# comment out if building for the first time

build_json(f"{base}/mix_clean", f"{out_path}/mix_clean.json")
build_json(f"{base}/s1", f"{out_path}/s1.json")
build_json(f"{base}/s2", f"{out_path}/s2.json")
build_json(f"{base}/s3", f"{out_path}/s3.json")

In [ ]:
base = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/test"
out_path = "/kaggle/working/json_dir/test"

# comment out if building for the first time

build_json(f"{base}/mix_clean", f"{out_path}/mix_clean.json")
build_json(f"{base}/s1", f"{out_path}/s1.json")
build_json(f"{base}/s2", f"{out_path}/s2.json")
build_json(f"{base}/s3", f"{out_path}/s3.json")

In [5]:
import os
import json
from tkinter.tix import Tree
import numpy as np
from typing import Any, Tuple
import soundfile as sf
import torch
from pytorch_lightning import LightningDataModule
from pytorch_lightning.core.mixins import HyperparametersMixin
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from typing import Dict, Iterable, List, Iterator
from rich import print
from pytorch_lightning.utilities import rank_zero_only

/tmp/ipykernel_163/1705073945.py:3: DeprecationWarning: The Tix Tk extension is unmaintained, and the tkinter.tix wrapper module is deprecated in favor of tkinter.ttk
  from tkinter.tix import Tree


In [6]:
import math
import glob
import os

import numpy as np
import torchaudio
import torch as th
import torch.nn as nn
import torch.nn.functional as tf
import librosa.filters as filters

import librosa
import scipy as sp
import matplotlib.pyplot as plt
import IPython.display as ipd

from typing import Optional, Tuple
from packaging.version import Version

EPSILON = float(np.finfo(np.float32).eps)
TORCH_VERSION = th.__version__

if TORCH_VERSION >= "1.7":
    from torch.fft import fft as fft_func
    print(TORCH_VERSION)
else:
    pass

2.10.0+cu128

In [7]:
@rank_zero_only
def print_(message: str):
    print(message)


def normalize_tensor_wav(wav_tensor, eps=1e-6, std=None):
    mean = wav_tensor.mean(-1, keepdim=True)
    if std is None:
        std = wav_tensor.std(-1, keepdim=True)
    return (wav_tensor - mean) / (std + eps)



class Libri3MixDataset(Dataset):
    def __init__(
        self,
        json_dir: str = "/kaggle/working/",
        n_src: int = 3,
        sample_rate: int = 8000,
        fps: int = 25,
        segment: float = 4.0,
        normalize_audio: bool = False,
        audio_only: bool = True,
    ) -> None:
        super().__init__()
        self.EPS = 1e-8
        if json_dir is None:
            raise ValueError("JSON DIR is None!")
        if n_src < 1:
            raise ValueError("n_src must be >= 1, got {}".format(n_src))
        self.json_dir = json_dir
        self.sample_rate = sample_rate
        self.normalize_audio = normalize_audio
        self.audio_only = audio_only
        if segment is None:
            self.seg_len = None
            self.fps_len = None
        else:
            self.seg_len = int(segment * sample_rate)
            self.fps_len = int(segment * fps)
        self.n_src = n_src
        self.test = self.seg_len is None

        mix_json = os.path.join(json_dir, "mix_clean.json")
        # dynamically build s1.json ... sN.json instead of hardcoding names
        source_names = [f"s{i+1}" for i in range(n_src)]
        sources_json = [
            os.path.join(json_dir, name + ".json") for name in source_names
        ]

        with open(mix_json, "r") as f:
            mix_infos = json.load(f)
        sources_infos = []
        for src_json in sources_json:
            with open(src_json, "r") as f:
                sources_infos.append(json.load(f))

        self.mix = []
        self.sources = []

        if self.n_src == 1:
            # special case: each source becomes its own single-source example
            orig_len = len(mix_infos) * len(sources_infos)
            drop_utt, drop_len = 0, 0
            if not self.test:
                for i in range(len(mix_infos) - 1, -1, -1):
                    if mix_infos[i][1] < self.seg_len:
                        drop_utt += 1
                        drop_len += mix_infos[i][1]
                        del mix_infos[i]
                        for src_inf in sources_infos:
                            del src_inf[i]
                    else:
                        for src_inf in sources_infos:
                            self.mix.append(mix_infos[i])
                            self.sources.append(src_inf[i])
            else:
                for i in range(len(mix_infos)):
                    for src_inf in sources_infos:
                        self.mix.append(mix_infos[i])
                        self.sources.append(src_inf[i])

            print_(
                "Drop {} utts({:.2f} h) from {} (shorter than {} samples)".format(
                    drop_utt, drop_len / sample_rate / 3600, orig_len, self.seg_len
                )
            )
            self.length = len(self.mix)

        else:
            # n_src >= 2 (2, 3, 4, ...): utterances stay index-aligned across all sources
            orig_len = len(mix_infos)
            drop_utt, drop_len = 0, 0
            if not self.test:
                for i in range(len(mix_infos) - 1, -1, -1):  # go backward
                    if mix_infos[i][1] < self.seg_len:
                        drop_utt += 1
                        drop_len += mix_infos[i][1]
                        del mix_infos[i]
                        for src_inf in sources_infos:
                            del src_inf[i]

            print_(
                "Drop {} utts({:.2f} h) from {} (shorter than {} samples)".format(
                    drop_utt, drop_len / sample_rate / 3600, orig_len, self.seg_len
                )
            )
            self.mix = mix_infos
            self.sources = sources_infos
            self.length = len(self.mix)

    def __len__(self):
        return self.length

    def preprocess_audio_only(self, idx: int):
        if self.n_src == 1:
            if self.mix[idx][1] == self.seg_len or self.test:
                rand_start = 0
            else:
                rand_start = np.random.randint(0, self.mix[idx][1] - self.seg_len)
            stop = None if self.test else rand_start + self.seg_len

            x, _ = sf.read(
                self.mix[idx][0], start=rand_start, stop=stop, dtype="float32"
            )
            s, _ = sf.read(
                self.sources[idx][0], start=rand_start, stop=stop, dtype="float32"
            )

            target = torch.from_numpy(s)
            mixture = torch.from_numpy(x)
            if self.normalize_audio:
                m_std = mixture.std(-1, keepdim=True)
                mixture = normalize_tensor_wav(mixture, eps=self.EPS, std=m_std)
                target = normalize_tensor_wav(target, eps=self.EPS, std=m_std)
            return mixture, target.unsqueeze(0), self.mix[idx][0].split("/")[-1]

        # n_src >= 2 — generalized, works for any number of sources
        if self.mix[idx][1] == self.seg_len or self.test:
            rand_start = 0
        else:
            rand_start = np.random.randint(0, self.mix[idx][1] - self.seg_len)
        stop = None if self.test else rand_start + self.seg_len

        x, _ = sf.read(self.mix[idx][0], start=rand_start, stop=stop, dtype="float32")

        source_arrays = []
        for src in self.sources:
            s, _ = sf.read(src[idx][0], start=rand_start, stop=stop, dtype="float32")
            source_arrays.append(s)
        sources = torch.from_numpy(np.vstack(source_arrays))
        mixture = torch.from_numpy(x)

        if self.normalize_audio:
            m_std = mixture.std(-1, keepdim=True)
            mixture = normalize_tensor_wav(mixture, eps=self.EPS, std=m_std)
            sources = normalize_tensor_wav(sources, eps=self.EPS, std=m_std)

        return mixture, sources, self.mix[idx][0].split("/")[-1]

    def __getitem__(self, index: int):
        return self.preprocess_audio_only(index)

import pytorch_lightning as pl
class Libri3MixDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dir: str,
        valid_dir: str,
        test_dir: str,
        n_src: int = 3,
        sample_rate: int = 8000,
        fps: int = 25,
        segment: float = 4.0,
        normalize_audio: bool = False,
        batch_size: int = 64,
        num_workers: int = 0,
        pin_memory: bool = False,
        persistent_workers: bool = False,
        audio_only: bool = True,
    ) -> None:
        super().__init__()
        if train_dir == None or valid_dir == None or test_dir == None:
            raise ValueError("JSON DIR is None!")
        if n_src < 1:
            raise ValueError("n_src must be >= 1, got {}".format(n_src))
        # this line allows to access init params with 'self.hparams' attribute
        self.train_dir = train_dir
        self.valid_dir = valid_dir
        self.test_dir = test_dir
        self.n_src = n_src
        self.sample_rate = sample_rate
        self.fps = fps
        self.segment = segment
        self.normalize_audio = normalize_audio
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.persistent_workers = persistent_workers
        self.audio_only = audio_only
        self.data_train: Dataset = None
        self.data_val: Dataset = None
        self.data_test: Dataset = None

    def setup(self, stage=None):
        self.data_train = Libri3MixDataset(
            json_dir=self.train_dir,
            n_src=self.n_src,
            sample_rate=self.sample_rate,
            fps=self.fps,
            segment=self.segment,
            normalize_audio=self.normalize_audio,
            audio_only=self.audio_only,
        )
    
        self.data_val = Libri3MixDataset(
            json_dir=self.valid_dir,
            n_src=self.n_src,
            sample_rate=self.sample_rate,
            fps=self.fps,
            segment=self.segment,
            normalize_audio=self.normalize_audio,
            audio_only=self.audio_only,
        )
    
        self.data_test = Libri3MixDataset(
            json_dir=self.test_dir,
            n_src=self.n_src,
            sample_rate=self.sample_rate,
            fps=self.fps,
            segment=self.segment,
            normalize_audio=self.normalize_audio,
            audio_only=self.audio_only,
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            dataset=self.data_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=self.persistent_workers,
            pin_memory=self.pin_memory,
            drop_last=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            dataset=self.data_val,
            shuffle=False,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            persistent_workers=self.persistent_workers,
            pin_memory=self.pin_memory,
            drop_last=True,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            dataset=self.data_test,
            shuffle=False,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            persistent_workers=self.persistent_workers,
            pin_memory=self.pin_memory,
            drop_last=True,
        )

    @property
    def make_loader(self):
        return self.train_dataloader(), self.val_dataloader(), self.test_dataloader()

    @property
    def make_sets(self):
        return self.data_train, self.data_val, self.data_test

In [8]:
import os, json
import soundfile as sf

DATASET_ROOT = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min"
WORK_ROOT = "/kaggle/input/datasets/garvs777/checkpointss/checkpoints/json_dir"
N_SRC = 3
SPLITS = {
    "train-100": "train-100",
    "dev": "dev",
    "test": "test",
}

# def build_split_json(split_dir_name, out_dir):
#     os.makedirs(out_dir, exist_ok=True)
#     src_folder = os.path.join(DATASET_ROOT, split_dir_name)

#     def scan(folder_name):
#         folder = os.path.join(src_folder, folder_name)
#         entries = []
#         for fname in sorted(os.listdir(folder)):
#             if not fname.endswith(".wav"):
#                 continue
#             fpath = os.path.join(folder, fname)
#             info = sf.info(fpath)
#             entries.append([fpath, info.frames])
#         return entries

#     mix_entries = scan("mix_clean")
#     with open(os.path.join(out_dir, "mix_clean.json"), "w") as f:
#         json.dump(mix_entries, f)

#     for i in range(N_SRC):
#         s_entries = scan(f"s{i+1}")
#         with open(os.path.join(out_dir, f"s{i+1}.json"), "w") as f:
#             json.dump(s_entries, f)

#     print(f"[{split_dir_name}] mix={len(mix_entries)} utterances -> {out_dir}")

# for split_dir_name in SPLITS.values():
#     out_dir = os.path.join(WORK_ROOT, split_dir_name)
#     build_split_json(split_dir_name, out_dir)


In [9]:
dm = Libri3MixDataModule(
    train_dir=os.path.join(WORK_ROOT, "train-100"),
    valid_dir=os.path.join(WORK_ROOT, "dev"),
    test_dir=os.path.join(WORK_ROOT, "test"),
    n_src=N_SRC,
    sample_rate=8000,
    segment=4.0,          # 4-second training chunks
    batch_size=8,         # keep small — bump up once you confirm memory headroom
    num_workers=2,
)
dm.setup()

_train_loader = dm.train_dataloader()
_mixture, _sources, _names = next(iter(_train_loader))
print("mixture:", _mixture.shape)   # (B, T)
print("sources:", _sources.shape)   # (B, n_src, T)
print("example name:", _names[0])


Drop 669 utts(0.65 h) from 9300 (shorter than 32000 samples)

Drop 1265 utts(1.21 h) from 3000 (shorter than 32000 samples)

Drop 1515 utts(1.45 h) from 3000 (shorter than 32000 samples)

mixture:
torch.Size([8, 32000])

sources:
torch.Size([8, 3, 32000])

example name: 441-128988-0008_2817-142380-0009_412-126975-0032.wav

In [10]:
import pytorch_lightning as pl

print(type(dm))
print(isinstance(dm, pl.LightningDataModule))

<class '__main__.Libri3MixDataModule'>

True

In [11]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.10.0+cu128

12.8

True

In [12]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Fri Jul 17 06:12:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0       

In [13]:
import torch, sys
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda (torch built with):", torch.version.cuda)
print("cxx11 ABI:", torch._C._GLIBCXX_USE_CXX11_ABI)

python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

torch: 2.10.0+cu128

cuda (torch built with): 12.8

cxx11 ABI: True

In [14]:
!pip install --no-index --find-links=/kaggle/input/datasets/garvs777/wheels/wheels --no-deps mamba-ssm causal-conv1d einops

Looking in links: /kaggle/input/datasets/garvs777/wheels/wheels
Processing /kaggle/input/datasets/garvs777/wheels/wheels/mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/garvs777/wheels/wheels/causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl


In [15]:
import torch
from mamba_ssm import Mamba

print("torch:", torch.__version__, "| cuda avail:", torch.cuda.is_available())

m = Mamba(d_model=16, d_state=16, d_conv=4, expand=2).to("cuda")
x = torch.randn(2, 64, 16, device="cuda")
y = m(x)
print("forward pass OK:", y.shape)

torch: 2.10.0+cu128 | cuda avail: True

forward pass OK:
torch.Size([2, 64, 16])

In [16]:
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print("mamba_ssm imported OK — using real Mamba blocks for FD/TD modules.")
except Exception as e:
    MAMBA_AVAILABLE = False
    print(f"mamba_ssm unavailable ({e!r}); falling back to BiGRU so the notebook still runs end-to-end.")


mamba_ssm imported OK — using real Mamba blocks for FD/TD modules.

In [17]:
class STFTEncoder(nn.Module):
    def __init__(self, n_fft=256, hop_length=64, win_length=256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", th.hann_window(win_length))

    def forward(self, wav):
        # wav: (B, T)
        spec = th.stft(
            wav, n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.win_length, window=self.window,
            return_complex=True, center=True,
        )  # (B, F, Tf) complex
        feat = th.stack([spec.real, spec.imag], dim=1)  # (B, 2, F, Tf)
        return feat, spec


class STFTDecoder(nn.Module):
    def __init__(self, n_fft=256, hop_length=64, win_length=256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", th.hann_window(win_length))

    def forward(self, complex_spec, length=None):
        # complex_spec: (B, F, Tf) complex
        return th.istft(
            complex_spec, n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.win_length, window=self.window,
            center=True, length=length,
        )


In [18]:
class BMamba(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        if MAMBA_AVAILABLE:
            self.fwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            self.bwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
            self._mode = "mamba"
        else:
            self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)
            self._mode = "gru"

    def forward(self, x):
        # x: (B, L, D)
        if self._mode == "mamba":
            fwd_out = self.fwd(x)
            bwd_out = self.bwd(x.flip(dims=[1])).flip(dims=[1])
            out = fwd_out + bwd_out
        else:
            out, _ = self.gru(x)
        return self.norm(out + x)


In [19]:
class FDModule(nn.Module):
    """Frequency-axis BMamba, applied independently at every time frame."""
    def __init__(self, d_model):
        super().__init__()
        self.bmamba = BMamba(d_model)

    def forward(self, x):
        # x: (B, D, F, Tf)
        B, D, F, Tf = x.shape
        x = x.permute(0, 3, 2, 1).reshape(B * Tf, F, D)
        x = self.bmamba(x)
        x = x.reshape(B, Tf, F, D).permute(0, 3, 2, 1)
        return x


class TDModule(nn.Module):
    """Time-axis BMamba, applied independently at every frequency bin. Replaces BLSTM."""
    def __init__(self, d_model):
        super().__init__()
        self.bmamba = BMamba(d_model)

    def forward(self, x):
        # x: (B, D, F, Tf)
        B, D, F, Tf = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * F, Tf, D)
        x = self.bmamba(x)
        x = x.reshape(B, F, Tf, D).permute(0, 3, 1, 2)
        return x


class TFAModule(nn.Module):
    """Pools each frame across frequency into one embedding, runs full self-attention
    across frames for global context, then broadcasts the result back."""
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.pool_proj = nn.Linear(d_model, d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, D, F, Tf)
        B, D, F, Tf = x.shape
        frame_emb = x.mean(dim=2).permute(0, 2, 1)     # (B, Tf, D) avg-pool over freq
        frame_emb = self.pool_proj(frame_emb)
        attn_out, _ = self.mha(frame_emb, frame_emb, frame_emb)
        attn_out = self.out_proj(attn_out)              # (B, Tf, D)
        attn_out = attn_out.permute(0, 2, 1).unsqueeze(2)  # (B, D, 1, Tf)
        x = x + attn_out                                 # broadcast across frequency
        x = self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        return x


class SPMambaBlock(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.fd = FDModule(d_model)
        self.td = TDModule(d_model)
        self.tfa = TFAModule(d_model, n_heads)

    def forward(self, x):
        x = self.fd(x)
        x = self.td(x)
        x = self.tfa(x)
        return x


In [20]:
class SPMambaSeparator(nn.Module):
    def __init__(self, n_src=3, d_model=64, n_blocks=4, n_fft=256, hop_length=64,
                 win_length=256, n_heads=4):
        super().__init__()
        self.n_src = n_src
        self.encoder = STFTEncoder(n_fft, hop_length, win_length)
        self.decoder = STFTDecoder(n_fft, hop_length, win_length)
        self.input_proj = nn.Conv2d(2, d_model, kernel_size=3, padding=1)
        self.input_norm = nn.GroupNorm(1, d_model)
        self.blocks = nn.ModuleList([SPMambaBlock(d_model, n_heads) for _ in range(n_blocks)])

        self.mask_head = nn.Conv2d(d_model, n_src * 2, kernel_size=3, padding=1)  # real+imag per slot
        self.presence_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, n_src),
        )

    def forward(self, wav):
        # wav: (B, T)
        feat, complex_spec = self.encoder(wav)     # feat (B,2,F,Tf), complex_spec (B,F,Tf)
        x = self.input_proj(feat)   # (B, D, F, Tf)
        x = self.input_norm(x) 
        for block in self.blocks:
            x = block(x)

        B, D, F, Tf = x.shape
        masks = self.mask_head(x).view(B, self.n_src, 2, F, Tf)
        masks = th.tanh(masks)
        mask_complex = th.complex(masks[:, :, 0], masks[:, :, 1])   # (B, n_src, F, Tf)

        sep_specs = mask_complex * complex_spec.unsqueeze(1)        # (B, n_src, F, Tf)

        sep_waves = th.stack(
            [self.decoder(sep_specs[:, i], length=wav.shape[-1]) for i in range(self.n_src)],
            dim=1,
        )  # (B, n_src, T)

        presence_logits = self.presence_head(x)   # (B, n_src)
        return sep_waves, presence_logits


In [ ]:
import itertools


def zero_mean(x):
    return x - x.mean(dim=-1, keepdim=True)


def si_sdr(est, target, eps=1e-8, zero_mean_norm=True, ratio_clamp=1e6):
    if zero_mean_norm:
        target = zero_mean(target)
        est = zero_mean(est)
    target_energy = th.sum(target ** 2, dim=-1, keepdim=True) + eps
    proj = th.sum(est * target, dim=-1, keepdim=True) * target / target_energy
    noise = est - proj
    ratio = th.sum(proj ** 2, dim=-1) / (th.sum(noise ** 2, dim=-1) + eps)
    ratio = th.clamp(ratio, min=eps, max=ratio_clamp)
    return 10 * th.log10(ratio)


def snr(est, target, eps=1e-8, zero_mean_norm=True, ratio_clamp=1e6):
    if zero_mean_norm:
        target = zero_mean(target)
        est = zero_mean(est)
    signal_energy = th.sum(target ** 2, dim=-1) + eps
    noise_energy = th.sum((est - target) ** 2, dim=-1) + eps
    ratio = th.clamp(signal_energy / noise_energy, min=eps, max=ratio_clamp)
    return 10 * th.log10(ratio)


def deterministic_energy_order(sources):
    energy = th.sum(sources ** 2, dim=-1)
    order = th.argsort(energy, dim=-1, descending=True)
    return th.gather(sources, 1, order.unsqueeze(-1).expand(-1, -1, sources.shape[-1]))


def deterministic_snr_loss(est_waves, sources):
    return -snr(est_waves, deterministic_energy_order(sources)).mean()


def deterministic_sisdr_loss(est_waves, sources):
    return -si_sdr(est_waves, deterministic_energy_order(sources)).mean()


def _pit_loss(est_waves, sources, metric_fn):
    n_src = est_waves.shape[1]
    best = None
    for perm in itertools.permutations(range(n_src)):
        score = metric_fn(est_waves, sources[:, perm, :]).mean(dim=-1)
        best = score if best is None else th.maximum(best, score)
    return -best.mean()


def pit_snr_loss(est_waves, sources):
    return _pit_loss(est_waves, sources, snr)


def pit_sisdr_loss(est_waves, sources):
    return _pit_loss(est_waves, sources, si_sdr)


def presence_bce_loss(presence_logits, n_active):
    B, n_src = presence_logits.shape
    idx = th.arange(n_src, device=presence_logits.device).unsqueeze(0).expand(B, -1)
    target = (idx < n_active.unsqueeze(1)).float()
    return tf.binary_cross_entropy_with_logits(presence_logits, target)

In [22]:
import pytorch_lightning as pl

class SPMambaLitModule(pl.LightningModule):
    def __init__(self, n_src=3, d_model=64, n_blocks=4, lr=1e-3,
                 assignment_mode="pit", presence_weight=1.0):
        super().__init__()
        self.save_hyperparameters()
        self.model = SPMambaSeparator(n_src=n_src, d_model=d_model, n_blocks=n_blocks)

    def forward(self, wav):
        return self.model(wav)

    def _step(self, batch, stage):
        mixture, sources, _ = batch
        est_waves, presence_logits = self.model(mixture)

        # train on SNR (matches original repo's pairwise_neg_snr training objective),
        # validate/report on SI-SDR (matches original repo's val/test metric)
        if stage == "train":
            sep_loss = (pit_snr_loss(est_waves, sources)
                        if self.hparams.assignment_mode == "pit"
                        else deterministic_snr_loss(est_waves, sources))
        else:
            sep_loss = (pit_sisdr_loss(est_waves, sources)
                        if self.hparams.assignment_mode == "pit"
                        else deterministic_sisdr_loss(est_waves, sources))

        n_active = th.full((mixture.shape[0],), self.hparams.n_src,
                            dtype=th.long, device=mixture.device)
        pres_loss = presence_bce_loss(presence_logits, n_active)

        loss = sep_loss + self.hparams.presence_weight * pres_loss
        self.log(f"{stage}_sep_loss", sep_loss, prog_bar=True, batch_size=mixture.shape[0])
        self.log(f"{stage}_presence_loss", pres_loss, prog_bar=True, batch_size=mixture.shape[0])
        self.log(f"{stage}_loss", loss, prog_bar=True, batch_size=mixture.shape[0])
        return loss

    def training_step(self, batch, batch_idx):
        loss = self._step(batch, "train")
        if not th.isfinite(loss):
            print(f"[warn] non-finite train loss at batch {batch_idx}, skipping step")
            return None
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        opt = th.optim.Adam(self.parameters(), lr=self.hparams.lr)
        sched = th.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=5
        )
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "monitor": "val_loss"},
        }

In [23]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.current_device() if torch.cuda.is_available() else "No GPU")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True

0

NVIDIA RTX PRO 6000 Blackwell Server Edition

In [24]:
batch = next(iter(dm.train_dataloader()))

print(type(batch))
print(len(batch))

for i, item in enumerate(batch):
    print(i, type(item))

<class 'list'>

3

0 <class 'torch.Tensor'>

1 <class 'torch.Tensor'>

2 <class 'tuple'>

In [25]:
print(batch[2])

(
    '1723-141149-0016_78-369-0012_4853-27671-0017.wav',
    '83-9960-0003_6385-220959-0008_328-129766-0001.wav',
    '4214-7146-0001_32-21634-0004_6385-34655-0021.wav',
    '4018-103416-0009_8975-270782-0002_1081-125237-0035.wav',
    '374-180298-0038_2092-145706-0012_250-142276-0031.wav',
    '5688-15787-0027_1263-139804-0019_8630-305212-0030.wav',
    '8425-287387-0005_5703-47198-0051_125-121124-0017.wav',
    '460-172359-0046_4813-248638-0044_8098-278278-0015.wav'
)

In [26]:
device = torch.device("cuda")

_sanity_model = SPMambaLitModule(
    n_src=N_SRC,
    d_model=64,
    n_blocks=2,
    assignment_mode="pit",
).to(device)

mixture, sources, filenames = next(iter(dm.train_dataloader()))

mixture = mixture.to(device)
sources = sources.to(device)

_loss = _sanity_model._step(
    (mixture, sources, filenames),
    "sanity"
)

print("Sanity loss:", _loss.item())

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


Sanity loss: 14.732611656188965

In [28]:
from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    dirpath="/kaggle/input/datasets/garvs777/checkpointss/checkpoints/lightning_logs/version_4/checkpoints",
    filename="epoch=6-step=7546",
    save_top_k=1,
    monitor="val_loss",
    mode="min",
    save_last=True,
)

In [ ]:
lit_model = SPMambaLitModule(
    n_src=N_SRC,
    d_model=128,
    n_blocks=6,
    lr=1e-3,
    assignment_mode="pit",   # or "pit" for the baseline run
    presence_weight=1.0,
)

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices="auto",
    precision="32" if th.cuda.is_available() else 32,
    log_every_n_steps=10,
    gradient_clip_val=5.0,
    callbacks=[checkpoint_cb],
)

trainer.fit(
    lit_model,
    train_dataloaders=dm.train_dataloader(),
    val_dataloaders=dm.val_dataloader(),
    ckpt_path="/kaggle/input/datasets/garvs777/checkpointss/checkpoints/lightning_logs/version_4/checkpoints/epoch=6-step=7546.ckpt"
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at /kaggle/input/datasets/garvs777/checkpointss/checkpoints/lightning_logs/version_4/checkpoints/epoch=6-step=7546.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ SPMambaSeparator │  3.4 M │ train │     0 │
└───┴───────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 3.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.4 M                                                                                                
Total estimated model params size (MB): 13.650                                                                     
Modules in train mode: 259                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restored all states from the checkpoint at /kaggle/input/datasets/garvs777/checkpointss/checkpoints/lightning_logs/version_4/checkpoints/epoch=6-step=7546.ckpt


Output()

In [28]:
lit_model = lit_model.cuda()
lit_model.eval()
_val_batch = next(iter(dm.test_dataloader()))
_mix, _src, _names = _val_batch
_mix, _src = _mix.to(lit_model.device), _src.to(lit_model.device)

with th.no_grad():
    _est_waves, _presence_logits = lit_model.model(_mix)
    _presence_probs = th.sigmoid(_presence_logits)

    # find the best permutation for THIS example, same way pit_sisdr_loss does
    n_src = _est_waves.shape[1]
    best_perm, best_score = None, None
    for perm in itertools.permutations(range(n_src)):
        score = si_sdr(_est_waves[0:1], _src[0:1, perm, :]).mean()
        if best_score is None or score > best_score:
            best_score, best_perm = score, perm

_src_aligned = _src[:, best_perm, :]   # reorder ground truth to match model's output order

print("Example:", _names[0], "| best perm:", best_perm)
ipd.display(ipd.Audio(_mix[0].cpu().numpy(), rate=8000))
for i in range(N_SRC):
    print(f"Ground-truth (aligned) speaker {i+1}:")
    ipd.display(ipd.Audio(_src_aligned[0, i].cpu().numpy(), rate=8000))
    print(f"Predicted slot {i+1} (presence={_presence_probs[0, i].item():.2f}):")
    ipd.display(ipd.Audio(_est_waves[0, i].cpu().numpy(), rate=8000))

Example: 1089-134686-0000_2094-142345-0037_4446-2271-0014.wav | best perm:
(1, 0, 2)

Ground-truth (aligned) speaker 1:

Predicted slot 1 (presence=1.00):

Ground-truth (aligned) speaker 2:

Predicted slot 2 (presence=1.00):

Ground-truth (aligned) speaker 3:

Predicted slot 3 (presence=1.00):

In [31]:
# Save (after training)
torch.save({
    "state_dict": lit_model.model.state_dict(),   # just the SPMambaSeparator weights
    "config": {
        "n_src": 3,
        "d_model": 128,
        "n_blocks": 6,
    },
}, "spmamba_checkpoint.pt")

In [33]:
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

In [35]:
import shutil
shutil.make_archive('/kaggle/working/checkpoints', 'zip', '/kaggle/working/')

'/kaggle/working/checkpoints.zip'